In [ ]:
# ============================================================
#  STUDENT DROPOUT & ACADEMIC SUCCESS — KAGGLE EDA
#  Dataset : Student Dropout & Academic Success
#  Purpose : Full Exploratory Data Analysis (EDA)
# ============================================================

# ── IMPORTS ─────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.ticker import PercentFormatter
import seaborn as sns
from scipy import stats

# ── CUSTOM PALETTE (Fresh Mint-Lavender-Peach palette) ──────
# A unique pastel-botanical palette rarely seen on Kaggle
PALETTE = {
    "Graduate" : "#6BCBA5",   # mint green
    "Enrolled" : "#A78BDB",   # soft violet
    "Dropout"  : "#F4896B",   # coral peach
}
CAT_COLORS  = ["#6BCBA5", "#A78BDB", "#F4896B", "#F9C74F", "#90C0F8", "#F78CA0"]
SEQ_CMAP    = "YlGn"           # sequential
DIV_CMAP    = "RdYlGn"         # diverging
BG_COLOR    = "#F7F9FC"
GRID_COLOR  = "#E2E8F0"
TEXT_COLOR  = "#2D3748"

plt.rcParams.update({
    "figure.facecolor"  : BG_COLOR,
    "axes.facecolor"    : BG_COLOR,
    "axes.edgecolor"    : GRID_COLOR,
    "axes.labelcolor"   : TEXT_COLOR,
    "axes.grid"         : True,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "grid.color"        : GRID_COLOR,
    "grid.linestyle"    : "--",
    "grid.alpha"        : 0.7,
    "xtick.color"       : TEXT_COLOR,
    "ytick.color"       : TEXT_COLOR,
    "font.family"       : "DejaVu Sans",
    "font.size"         : 11,
    "figure.dpi"        : 130,
})

TARGET_ORDER = ["Graduate", "Enrolled", "Dropout"]

# ════════════════════════════════════════════════════════════
# 1 ▸  LOAD & DESCRIBE DATA
# ════════════════════════════════════════════════════════════
print("=" * 65)
print("  STUDENT DROPOUT & ACADEMIC SUCCESS — EDA")
print("=" * 65)

# ----------------------------------------------------------
# NOTE FOR KAGGLE:
#   If running on Kaggle, replace the path below with:
#   df = pd.read_csv("/kaggle/input/<your-dataset>/student_dropout_academic_success_csv.csv", sep=";")
# ----------------------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/hamnawaseem112222222/predict-students-dropout-and-academic-success/student_dropout_academic_success.csv.csv", sep=";")

# Clean up stray whitespace in column names
df.columns = df.columns.str.strip()

print(f"\n📌 Dataset shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"📌 Target classes : {df['Target'].value_counts().to_dict()}")
print(f"📌 Missing values : {df.isnull().sum().sum()} total\n")

print("─" * 65)
print("DATASET DESCRIPTION")
print("─" * 65)
description = """
This dataset originates from a Portuguese higher-education institution and
tracks 4424 students across 36 features.  The goal is to predict — at
enrolment time — whether a student will eventually Graduate, remain Enrolled,
or Drop Out.

Feature groups
 • Demographics        – gender, age at enrolment, nationality, marital status
 • Socio-economic      – parents' education & occupation, debtor flag,
                         scholarship, tuition-fee payment status
 • Academic background – previous qualification & grade, admission grade
 • Semester performance– credited / enrolled / evaluated / approved units,
                         average grade for semesters 1 & 2
 • Macro-economic      – unemployment rate, inflation rate, GDP at enrolment

Target distribution
 • Graduate  : 2 209  (49.9 %)
 • Dropout   : 1 421  (32.1 %)
 • Enrolled  :   794  (17.9 %)
"""
print(description)

# ── Summary Statistics ───────────────────────────────────────
print("─" * 65)
print("SUMMARY STATISTICS (numeric columns)")
print("─" * 65)
numeric_summary = df.describe().T
numeric_summary["missing"] = df.isnull().sum()
numeric_summary["dtype"]   = df.dtypes
print(numeric_summary.to_string())
print()


# ════════════════════════════════════════════════════════════
# 2 ▸  HELPER
# ════════════════════════════════════════════════════════════
def annotate_bars(ax, fmt="{:.1f}%", multiplier=100, fontsize=9):
    """Add percentage labels to bar charts."""
    total = sum(p.get_height() for p in ax.patches)
    for p in ax.patches:
        val = p.get_height()
        if val > 0:
            ax.annotate(
                fmt.format(val * multiplier / total),
                (p.get_x() + p.get_width() / 2., val),
                ha="center", va="bottom", fontsize=fontsize,
                color=TEXT_COLOR, fontweight="bold",
            )

def add_subtitle(ax, text, y=1.02):
    ax.annotate(text, xy=(0.5, y), xycoords="axes fraction",
                ha="center", fontsize=9, color="#718096", style="italic")

def target_colors(series):
    return [PALETTE.get(t, "#CCCCCC") for t in series]


# ════════════════════════════════════════════════════════════
# 3 ▸  CHART 1 — TARGET DISTRIBUTION (donut + bar)
# ════════════════════════════════════════════════════════════
print("Rendering Chart 1 : Target Distribution …")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle("Student Outcome Distribution", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

counts   = df["Target"].value_counts().reindex(TARGET_ORDER)
pcts     = counts / counts.sum() * 100
colors_t = [PALETTE[t] for t in TARGET_ORDER]

# — Donut —
wedges, texts, autotexts = axes[0].pie(
    counts, labels=TARGET_ORDER, colors=colors_t,
    autopct="%1.1f%%", startangle=90, pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2.5),
    textprops={"color": TEXT_COLOR},
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold")
axes[0].set_title("Proportion of Each Outcome", fontsize=12,
                  color=TEXT_COLOR, fontweight="bold", pad=12)

# — Horizontal bar —
bars = axes[1].barh(TARGET_ORDER, pcts, color=colors_t,
                    height=0.5, edgecolor="white", linewidth=1.5)
for bar, pct in zip(bars, pcts):
    axes[1].text(pct + 0.5, bar.get_y() + bar.get_height() / 2,
                 f"{pct:.1f}%", va="center", fontsize=10,
                 fontweight="bold", color=TEXT_COLOR)
axes[1].set_xlabel("Percentage (%)")
axes[1].set_title("Outcome Breakdown (%)", fontsize=12,
                  color=TEXT_COLOR, fontweight="bold", pad=12)
axes[1].set_xlim(0, 65)

plt.tight_layout()
plt.savefig("01_target_distribution.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Graduates are the majority class (≈50%), suggesting most students complete
  their degree. However, Dropouts represent ~32% — roughly 1 in 3 students —
  signalling a significant retention challenge. The Enrolled class (18%)
  represents students still in progress at observation time.
""")


# ════════════════════════════════════════════════════════════
# 4 ▸  CHART 2 — MISSING VALUES HEATMAP
# ════════════════════════════════════════════════════════════
print("Rendering Chart 2 : Missing Values …")

missing = df.isnull().sum()
if missing.sum() == 0:
    print("  ✔ No missing values detected — skipping heatmap.\n")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
                cmap="YlOrRd", ax=ax)
    ax.set_title("Missing Value Map", fontsize=14, fontweight="bold",
                 color=TEXT_COLOR)
    plt.tight_layout()
    plt.savefig("02_missing_values.png", bbox_inches="tight", dpi=130)
    plt.show()


# ════════════════════════════════════════════════════════════
# 5 ▸  CHART 3 — GENDER × TARGET
# ════════════════════════════════════════════════════════════
print("Rendering Chart 3 : Gender vs Target …")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle("Gender vs Student Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

gender_map  = {0: "Female", 1: "Male"}
df["Gender_label"] = df["Gender"].map(gender_map)
ct           = pd.crosstab(df["Gender_label"], df["Target"])
ct_pct       = ct.div(ct.sum(axis=1), axis=0) * 100

# Grouped bar
ct[TARGET_ORDER].plot(
    kind="bar", ax=axes[0], color=colors_t, edgecolor="white",
    linewidth=1.5, width=0.65)
axes[0].set_title("Count by Gender", fontsize=12,
                  color=TEXT_COLOR, fontweight="bold")
axes[0].set_xlabel("Gender"); axes[0].set_ylabel("Count")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(title="Outcome", loc="upper right")

# Stacked %
ct_pct[TARGET_ORDER].plot(
    kind="bar", stacked=True, ax=axes[1],
    color=colors_t, edgecolor="white", linewidth=1.5, width=0.55)
axes[1].set_title("Outcome Proportion by Gender", fontsize=12,
                  color=TEXT_COLOR, fontweight="bold")
axes[1].set_xlabel("Gender"); axes[1].set_ylabel("Percentage (%)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].yaxis.set_major_formatter(PercentFormatter())
axes[1].legend(title="Outcome", loc="lower right")

plt.tight_layout()
plt.savefig("03_gender_vs_target.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Male students show a noticeably higher dropout rate compared to females.
  Female students have a higher graduation proportion. This suggests gender
  plays a meaningful role in academic persistence and may reflect different
  support needs across genders.
""")


# ════════════════════════════════════════════════════════════
# 6 ▸  CHART 4 — AGE AT ENROLMENT DISTRIBUTION
# ════════════════════════════════════════════════════════════
print("Rendering Chart 4 : Age at Enrolment …")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Age at Enrolment vs Student Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

# KDE per outcome
for target, col in PALETTE.items():
    subset = df[df["Target"] == target]["Age at enrollment"]
    subset.plot.kde(ax=axes[0], color=col, lw=2.5, label=target)
    axes[0].axvline(subset.median(), color=col, lw=1.2, ls="--", alpha=0.7)

axes[0].set_title("Age Distribution (KDE)", fontsize=12,
                  color=TEXT_COLOR, fontweight="bold")
axes[0].set_xlabel("Age at Enrolment")
axes[0].set_ylabel("Density")
axes[0].legend(title="Outcome")
axes[0].set_xlim(15, 65)

# Violin
violin_data = [df[df["Target"] == t]["Age at enrollment"] for t in TARGET_ORDER]
vp = axes[1].violinplot(violin_data, positions=[1, 2, 3],
                        showmedians=True, showextrema=True)
for patch, col in zip(vp["bodies"], colors_t):
    patch.set_facecolor(col); patch.set_alpha(0.8)
vp["cmedians"].set_color(TEXT_COLOR); vp["cmedians"].set_linewidth(2)
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(TARGET_ORDER)
axes[1].set_title("Age Violin Plot by Outcome", fontsize=12,
                  color=TEXT_COLOR, fontweight="bold")
axes[1].set_ylabel("Age at Enrolment")

plt.tight_layout()
plt.savefig("04_age_vs_target.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Most students enrol in their late teens or early twenties. Dropout students
  have a wider, right-skewed age distribution — older enrollees are
  disproportionately more likely to drop out, possibly due to competing work
  or family responsibilities. Graduates cluster tightly around age 18-21.
""")


# ════════════════════════════════════════════════════════════
# 7 ▸  CHART 5 — SCHOLARSHIP & TUITION vs TARGET
# ════════════════════════════════════════════════════════════
print("Rendering Chart 5 : Scholarship & Tuition vs Target …")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Financial Indicators vs Student Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

for ax, col, title in zip(
    axes,
    ["Scholarship holder", "Tuition fees up to date"],
    ["Scholarship Status", "Tuition Fees Up to Date"]
):
    ct  = pd.crosstab(df[col], df["Target"])[TARGET_ORDER]
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.index = ["No", "Yes"]

    ct_pct.plot(kind="bar", stacked=True, ax=ax,
                color=colors_t, edgecolor="white", linewidth=1.5, width=0.5)
    ax.set_title(title, fontsize=12, color=TEXT_COLOR, fontweight="bold")
    ax.set_xlabel(""); ax.set_ylabel("Percentage (%)")
    ax.set_xticklabels(["No", "Yes"], rotation=0)
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.legend(title="Outcome", loc="lower right")

plt.tight_layout()
plt.savefig("05_financial_vs_target.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Students with scholarships graduate at far higher rates and rarely drop out,
  confirming financial aid as a protective factor. Similarly, students who are
  up to date with tuition fees have much higher graduation rates. Financial
  stress is a major predictor of dropout risk.
""")


# ════════════════════════════════════════════════════════════
# 8 ▸  CHART 6 — 1ST & 2ND SEMESTER GRADES
# ════════════════════════════════════════════════════════════
print("Rendering Chart 6 : Semester Grades vs Target …")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Semester Academic Grades vs Student Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

for ax, col, sem in zip(
    axes,
    ["Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)"],
    ["1st Semester", "2nd Semester"]
):
    for target, col_color in PALETTE.items():
        subset = df[(df["Target"] == target) & (df[col] > 0)][col]
        subset.plot.kde(ax=ax, color=col_color, lw=2.5, label=target)
    ax.set_title(f"{sem} Grade Distribution", fontsize=12,
                 color=TEXT_COLOR, fontweight="bold")
    ax.set_xlabel("Grade (0–20 scale)")
    ax.set_ylabel("Density")
    ax.legend(title="Outcome")
    ax.set_xlim(0, 20)

plt.tight_layout()
plt.savefig("06_grades_vs_target.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Graduates concentrate at higher grades (12-16) in both semesters, while
  Dropout students cluster at very low or zero grades, indicating early
  academic failure. Enrolled students span a broad range. Semester grades
  are among the strongest early-warning signals for dropout risk.
""")


# ════════════════════════════════════════════════════════════
# 9 ▸  CHART 7 — CURRICULAR UNITS APPROVED (Sem 1 & 2)
# ════════════════════════════════════════════════════════════
print("Rendering Chart 7 : Approved Units vs Target …")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Curricular Units Approved vs Student Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

for ax, col, sem in zip(
    axes,
    ["Curricular units 1st sem (approved)", "Curricular units 2nd sem (approved)"],
    ["1st Semester", "2nd Semester"]
):
    ax.set_facecolor(BG_COLOR)
    bp = ax.boxplot(
        [df[df["Target"] == t][col] for t in TARGET_ORDER],
        patch_artist=True,
        medianprops=dict(color=TEXT_COLOR, linewidth=2),
        whiskerprops=dict(color="#B0BEC5"),
        capprops=dict(color="#B0BEC5"),
        flierprops=dict(marker="o", color="#CBD5E0",
                        markersize=3, alpha=0.4),
    )
    for patch, col_color in zip(bp["boxes"], colors_t):
        patch.set_facecolor(col_color)
        patch.set_alpha(0.85)
        patch.set_linewidth(1.5)
    ax.set_xticklabels(TARGET_ORDER)
    ax.set_title(f"Approved Units — {sem}", fontsize=12,
                 color=TEXT_COLOR, fontweight="bold")
    ax.set_ylabel("Number of Approved Units")

plt.tight_layout()
plt.savefig("07_approved_units_vs_target.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Graduates pass significantly more curricular units each semester compared to
  Dropouts, who often approve zero units. This is the clearest separating
  feature: a student who fails to pass any units in Semester 1 or 2 is at
  extreme dropout risk.
""")


# ════════════════════════════════════════════════════════════
# 10 ▸  CHART 8 — DEBTOR & DISPLACED STATUS
# ════════════════════════════════════════════════════════════
print("Rendering Chart 8 : Debtor & Displaced vs Target …")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Student Background Flags vs Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

for ax, col, title in zip(
    axes,
    ["Debtor", "Displaced"],
    ["Debtor Status", "Displaced Student"]
):
    ct     = pd.crosstab(df[col], df["Target"])[TARGET_ORDER]
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.index = ["No", "Yes"]

    ct_pct.plot(kind="bar", ax=ax, color=colors_t,
                edgecolor="white", linewidth=1.5, width=0.5)
    ax.set_title(title, fontsize=12, color=TEXT_COLOR, fontweight="bold")
    ax.set_xticklabels(["No", "Yes"], rotation=0)
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set_ylabel("Percentage (%)")
    ax.legend(title="Outcome")

plt.tight_layout()
plt.savefig("08_debtor_displaced_vs_target.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Debtors show dramatically higher dropout rates — a strong signal of financial
  hardship leading to early departure. Displaced students (those who relocated
  for university) tend to graduate at slightly higher rates, possibly reflecting
  greater commitment, though they also have a notable dropout share.
""")


# ════════════════════════════════════════════════════════════
# 11 ▸  CHART 9 — MACRO-ECONOMIC INDICATORS
# ════════════════════════════════════════════════════════════
print("Rendering Chart 9 : Macro-Economic Indicators …")

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
fig.suptitle("Macro-Economic Context at Enrolment vs Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

macro_cols  = ["Unemployment rate", "Inflation rate", "GDP"]
macro_units = ["Rate (%)", "Rate (%)", "Index"]

for ax, col, unit in zip(axes, macro_cols, macro_units):
    for target, col_color in PALETTE.items():
        subset = df[df["Target"] == target][col]
        subset.plot.kde(ax=ax, color=col_color, lw=2.5, label=target)
        ax.axvline(subset.median(), color=col_color,
                   lw=1.2, ls=":", alpha=0.8)
    ax.set_title(col, fontsize=11, color=TEXT_COLOR, fontweight="bold")
    ax.set_xlabel(unit)
    ax.set_ylabel("Density")
    ax.legend(title="Outcome", fontsize=8)

plt.tight_layout()
plt.savefig("09_macro_economic.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Macro-economic conditions at the time of enrolment show subtle differences
  across outcomes. Students who enrolled during higher unemployment periods
  have a marginally elevated dropout risk, reflecting the broader economic
  pressures affecting students. GDP and inflation show less dramatic
  differences, suggesting micro-level factors (finances, grades) are stronger
  predictors.
""")


# ════════════════════════════════════════════════════════════
# 12 ▸  CHART 10 — CORRELATION HEATMAP
# ════════════════════════════════════════════════════════════
print("Rendering Chart 10 : Correlation Heatmap …")

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[numeric_cols].corr()

# Keep only columns most correlated with grade / approval
key_cols = [
    "Age at enrollment", "Admission grade",
    "Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)",
    "Scholarship holder", "Debtor", "Tuition fees up to date",
    "Unemployment rate", "GDP",
]
key_cols = [c for c in key_cols if c in corr.columns]
corr_sub = corr.loc[key_cols, key_cols]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_sub, dtype=bool))
sns.heatmap(
    corr_sub, mask=mask, ax=ax,
    cmap=DIV_CMAP, center=0,
    annot=True, fmt=".2f", annot_kws={"size": 9},
    linewidths=0.4, linecolor=GRID_COLOR,
    cbar_kws={"shrink": 0.7},
)
ax.set_title("Feature Correlation Heatmap (Key Variables)", fontsize=15,
             fontweight="bold", color=TEXT_COLOR, pad=14)
plt.tight_layout()
plt.savefig("10_correlation_heatmap.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  A strong positive correlation exists between 1st and 2nd semester approved
  units and grades — students who perform well early tend to continue
  performing well. Scholarship holder and Tuition fees up to date correlate
  positively with academic performance. Age at enrolment shows a weak negative
  correlation with academic success metrics.
""")


# ════════════════════════════════════════════════════════════
# 13 ▸  CHART 11 — ADMISSION GRADE vs SEMESTER GRADE (scatter)
# ════════════════════════════════════════════════════════════
print("Rendering Chart 11 : Admission Grade vs Semester Performance …")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Admission Grade vs Semester Performance by Outcome", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

for ax, col, sem in zip(
    axes,
    ["Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)"],
    ["1st Semester", "2nd Semester"]
):
    for target, col_color in PALETTE.items():
        sub = df[(df["Target"] == target) & (df[col] > 0)]
        ax.scatter(sub["Admission grade"], sub[col],
                   color=col_color, alpha=0.25, s=12, label=target)
    ax.set_xlabel("Admission Grade")
    ax.set_ylabel(f"{sem} Grade")
    ax.set_title(f"Admission → {sem} Grade", fontsize=12,
                 color=TEXT_COLOR, fontweight="bold")
    handles = [mpatches.Patch(color=PALETTE[t], label=t) for t in TARGET_ORDER]
    ax.legend(handles=handles, title="Outcome", fontsize=9)

plt.tight_layout()
plt.savefig("11_admission_vs_semester_grade.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Graduates cluster in the upper-right quadrant — high admission grades
  predict high semester performance. Dropout students populate the lower
  region irrespective of admission grade, implying that some academically
  capable students still drop out due to non-academic factors.
""")


# ════════════════════════════════════════════════════════════
# 14 ▸  CHART 12 — PARENTS' EDUCATION vs DROPOUT RATE
# ════════════════════════════════════════════════════════════
print("Rendering Chart 12 : Parental Education vs Dropout Rate …")

edu_map = {
    1: "Secondary", 2: "Higher (BSc)", 3: "Higher (MSc)",
    4: "Higher (PhD)", 5: "Freq. Higher", 6: "12th Year",
    9: "Basic 3rd Cycle", 10: "Basic 2nd Cycle",
    11: "Basic 1st Cycle", 12: "6th Year", 14: "10th Year",
    18: "General Commerce", 19: "10th Year–Tech",
    22: "Tech-Prof Course", 26: "7th Year (Old)",
    27: "Other", 29: "2nd Year Comp.", 30: "11th Year",
    34: "Unknown", 35: "Cant Read/Write",
    36: "Read w/o 4th Yr", 37: "Basic Education (4th Yr)",
    38: "Prep. Primary", 39: "Basic (6th Yr)",
    40: "Technological spec.", 41: "Higher Education",
    42: "Higher (Spec.)", 43: "Prof. Higher (Spec.)",
    44: "Masters (2nd Cycle)"
}

df["Mother_edu"]  = df["Mother's qualification"].map(edu_map).fillna("Other")
df["Father_edu"]  = df["Father's qualification"].map(edu_map).fillna("Other")
df["is_dropout"]  = (df["Target"] == "Dropout").astype(int)

top_n = 10
for parent, col_edu in [("Mother", "Mother_edu"), ("Father", "Father_edu")]:
    freq  = df[col_edu].value_counts().head(top_n).index
    rates = (df[df[col_edu].isin(freq)]
               .groupby(col_edu)["is_dropout"]
               .mean()
               .reindex(freq)
               .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Parental Education Level vs Dropout Rate", fontsize=16,
             fontweight="bold", color=TEXT_COLOR, y=1.02)

for ax, parent, col_edu in zip(axes,
                                ["Mother", "Father"],
                                ["Mother_edu", "Father_edu"]):
    freq  = df[col_edu].value_counts().head(top_n).index
    rates = (df[df[col_edu].isin(freq)]
               .groupby(col_edu)["is_dropout"]
               .mean()
               .reindex(freq)
               .sort_values(ascending=False))
    bars = ax.barh(rates.index, rates.values * 100,
                   color=CAT_COLORS[:len(rates)],
                   edgecolor="white", linewidth=1.5, height=0.6)
    for bar, val in zip(bars, rates.values * 100):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=9,
                color=TEXT_COLOR, fontweight="bold")
    ax.set_title(f"{parent}'s Education vs Dropout Rate", fontsize=12,
                 color=TEXT_COLOR, fontweight="bold")
    ax.set_xlabel("Dropout Rate (%)")
    ax.set_xlim(0, 70)

plt.tight_layout()
plt.savefig("12_parental_education_dropout.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Students whose parents have lower educational qualifications (basic or no
  schooling) exhibit higher dropout rates. Conversely, students from families
  where parents hold higher degrees (BSc, MSc, PhD) have notably lower dropout
  rates. This generational educational factor is a well-known socio-economic
  predictor in academic persistence research.
""")


# ════════════════════════════════════════════════════════════
# 15 ▸  CHART 13 — ENROLLMENT TYPE & DAYTIME vs EVENING
# ════════════════════════════════════════════════════════════
print("Rendering Chart 13 : Daytime vs Evening Attendance …")

fig, ax = plt.subplots(figsize=(9, 5.5))
attendance_col = "Daytime/evening attendance\t"
if attendance_col not in df.columns:
    attendance_col = [c for c in df.columns if "attendance" in c.lower()][0]

df["Attendance"] = df[attendance_col].map({1: "Daytime", 0: "Evening"})
ct     = pd.crosstab(df["Attendance"], df["Target"])[TARGET_ORDER]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

ct_pct.plot(kind="bar", stacked=True, ax=ax,
            color=colors_t, edgecolor="white", linewidth=1.5, width=0.45)
ax.set_title("Daytime vs Evening Attendance — Outcome Share", fontsize=14,
             fontweight="bold", color=TEXT_COLOR)
ax.set_xlabel("Attendance Type")
ax.set_ylabel("Percentage (%)")
ax.set_xticklabels(["Daytime", "Evening"], rotation=0)
ax.yaxis.set_major_formatter(PercentFormatter())
ax.legend(title="Outcome", loc="lower right")

plt.tight_layout()
plt.savefig("13_attendance_type.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  Daytime students exhibit a higher graduation rate than evening students.
  Evening attendance likely correlates with part-time employment, which can
  increase dropout risk. Institutions should provide targeted support for
  evening-program students.
""")


# ════════════════════════════════════════════════════════════
# 16 ▸  CHART 14 — OVERVIEW DASHBOARD (summary)
# ════════════════════════════════════════════════════════════
print("Rendering Chart 14 : Summary Dashboard …")

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Student Dropout & Academic Success — Summary Dashboard",
             fontsize=18, fontweight="bold", color=TEXT_COLOR, y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── Panel A : Target distribution ──
ax_a = fig.add_subplot(gs[0, 0])
counts_series = df["Target"].value_counts().reindex(TARGET_ORDER)
ax_a.bar(TARGET_ORDER, counts_series, color=colors_t,
         edgecolor="white", linewidth=2, width=0.5)
for i, (val, tgt) in enumerate(zip(counts_series, TARGET_ORDER)):
    ax_a.text(i, val + 30, str(val), ha="center", va="bottom",
              fontsize=10, fontweight="bold", color=TEXT_COLOR)
ax_a.set_title("Outcome Count", fontsize=11, fontweight="bold", color=TEXT_COLOR)
ax_a.set_ylabel("Count")

# ── Panel B : Scholarship impact ──
ax_b = fig.add_subplot(gs[0, 1])
sch_rate = (df.groupby("Scholarship holder")["is_dropout"].mean() * 100)
sch_rate.index = ["No Scholarship", "Scholarship"]
ax_b.bar(sch_rate.index, sch_rate.values,
         color=[PALETTE["Dropout"], PALETTE["Graduate"]],
         edgecolor="white", linewidth=2, width=0.45)
for i, val in enumerate(sch_rate.values):
    ax_b.text(i, val + 0.5, f"{val:.1f}%", ha="center",
              fontsize=10, fontweight="bold", color=TEXT_COLOR)
ax_b.set_title("Dropout Rate: Scholarship", fontsize=11,
               fontweight="bold", color=TEXT_COLOR)
ax_b.set_ylabel("Dropout Rate (%)")

# ── Panel C : Avg 1st Sem Grade by Target ──
ax_c = fig.add_subplot(gs[0, 2])
avg_grade = (df[df["Curricular units 1st sem (grade)"] > 0]
             .groupby("Target")["Curricular units 1st sem (grade)"]
             .mean()
             .reindex(TARGET_ORDER))
ax_c.bar(TARGET_ORDER, avg_grade, color=colors_t,
         edgecolor="white", linewidth=2, width=0.5)
for i, val in enumerate(avg_grade):
    ax_c.text(i, val + 0.1, f"{val:.2f}", ha="center",
              fontsize=10, fontweight="bold", color=TEXT_COLOR)
ax_c.set_title("Avg 1st Sem Grade", fontsize=11,
               fontweight="bold", color=TEXT_COLOR)
ax_c.set_ylabel("Average Grade (0–20)")

# ── Panel D : Avg Approved Units by Target ──
ax_d = fig.add_subplot(gs[1, 0])
avg_units = (df.groupby("Target")["Curricular units 1st sem (approved)"]
             .mean()
             .reindex(TARGET_ORDER))
ax_d.bar(TARGET_ORDER, avg_units, color=colors_t,
         edgecolor="white", linewidth=2, width=0.5)
for i, val in enumerate(avg_units):
    ax_d.text(i, val + 0.1, f"{val:.1f}", ha="center",
              fontsize=10, fontweight="bold", color=TEXT_COLOR)
ax_d.set_title("Avg Approved Units (Sem 1)", fontsize=11,
               fontweight="bold", color=TEXT_COLOR)
ax_d.set_ylabel("Units Approved")

# ── Panel E : Debtor dropout rate ──
ax_e = fig.add_subplot(gs[1, 1])
debt_rate = df.groupby("Debtor")["is_dropout"].mean() * 100
debt_rate.index = ["Not Debtor", "Debtor"]
ax_e.bar(debt_rate.index, debt_rate.values,
         color=[PALETTE["Graduate"], PALETTE["Dropout"]],
         edgecolor="white", linewidth=2, width=0.45)
for i, val in enumerate(debt_rate.values):
    ax_e.text(i, val + 0.5, f"{val:.1f}%", ha="center",
              fontsize=10, fontweight="bold", color=TEXT_COLOR)
ax_e.set_title("Dropout Rate: Debt Status", fontsize=11,
               fontweight="bold", color=TEXT_COLOR)
ax_e.set_ylabel("Dropout Rate (%)")

# ── Panel F : Age median by Target ──
ax_f = fig.add_subplot(gs[1, 2])
med_age = df.groupby("Target")["Age at enrollment"].median().reindex(TARGET_ORDER)
ax_f.bar(TARGET_ORDER, med_age, color=colors_t,
         edgecolor="white", linewidth=2, width=0.5)
for i, val in enumerate(med_age):
    ax_f.text(i, val + 0.1, f"{val:.0f} yrs", ha="center",
              fontsize=10, fontweight="bold", color=TEXT_COLOR)
ax_f.set_title("Median Age at Enrolment", fontsize=11,
               fontweight="bold", color=TEXT_COLOR)
ax_f.set_ylabel("Age (years)")

plt.savefig("14_summary_dashboard.png", bbox_inches="tight", dpi=130)
plt.show()

print("""
  ► INTERPRETATION
  The dashboard synthesises the most impactful findings:
  • Graduates score higher, pass more units, and are generally younger at enrolment.
  • Financial factors (scholarship, debt, tuition) strongly modulate dropout risk.
  • Median enrolment age for Dropouts exceeds that of Graduates.
""")


# ════════════════════════════════════════════════════════════
# 17 ▸  CONCLUSION
# ════════════════════════════════════════════════════════════
print("=" * 65)
print("  CONCLUSION")
print("=" * 65)
conclusion = """
Key Takeaways from the EDA
───────────────────────────────────────────────────────────────

1. ACADEMIC PERFORMANCE IS THE PRIMARY PREDICTOR
   Students who fail to pass curricular units in their first or
   second semester are overwhelmingly likely to drop out.  Semester
   grades and approved units are the most discriminating features.

2. FINANCIAL HARDSHIP AMPLIFIES DROPOUT RISK
   Debtors, students without scholarships, and those who are behind
   on tuition fees all drop out at substantially higher rates.
   Financial interventions (grants, debt counselling, flexible
   payment plans) could significantly reduce dropout numbers.

3. AGE & LIFE CIRCUMSTANCES MATTER
   Older enrollees — especially those juggling work and family — are
   more likely to leave before graduating.  Evening-programme students
   share a similar risk profile.

4. GENDER EFFECT IS REAL
   Male students are more likely to drop out; female students graduate
   at higher rates.  Targeted mentoring for male students could help.

5. PARENTAL EDUCATION AS BACKGROUND SIGNAL
   First-generation university students (parents with only basic
   schooling) face higher dropout rates, highlighting the importance
   of peer mentoring and additional academic support.

6. MACRO-ECONOMICS HAVE MODEST IMPACT
   Unemployment and GDP at enrolment time show subtle differences
   across outcomes.  While context matters, individual-level factors
   (grades, finances) are far more actionable for intervention.

RECOMMENDED EARLY-WARNING INDICATORS
   ✔ Curricular units approved Sem 1 < 3
   ✔ Semester grade < 10
   ✔ Student flagged as Debtor
   ✔ Tuition fees not up to date
   ✔ No scholarship AND enrolled in evening programme
   ✔ Age at enrolment > 25

These six flags, used together, could form a simple rule-based
early-warning system for academic advisors to flag at-risk students
within the first semester.
"""
print(conclusion)

print("\n✅  Analysis complete.  All charts saved as PNG files.")